# 1. Daten laden und bereinigen

Gruppe 05 (SC02_Gruppe_05): Dominik Erbsland, Joel Liechti. ZHAW, CAS Information Engineering, Scripting.

In diesem Notebook laden wir die Rohdaten, prüfen sie, bereinigen sie und ordnen jeder Kontrolle die Koordinaten der Haltestelle zu. Das Ergebnis speichern wir als CSV in `data/processed/`. Die Notebooks 2 und 3 arbeiten mit dieser Datei weiter.

**Reihenfolge:** Notebook 1, 2 und 3 nacheinander ausführen. Notebook 4 ist der Bericht.

In [ ]:
from pathlib import Path

import pandas as pd

# Pfade relativ zum Ordner notebooks/
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

## 1.1 Haltestellen einlesen

Die Haltestellen kommen von opendata.swiss (SBB). Ist die Datei noch nicht lokal vorhanden, laden wir sie einmal herunter und speichern sie in `data/raw/`.

`pd.read_csv` liest die Datei in ein **pandas DataFrame** (`stops_raw`). Daraus bauen wir ein zweites, kleineres DataFrame `stops` mit nur den Spalten, die wir brauchen.

In [ ]:
STOPS_FILE = RAW / "haltestelle_haltekante.csv"
STOPS_URL = "https://sbb.opendatasoft.com/api/v2/catalog/datasets/haltestelle-haltekante/exports/csv?use_labels=true"

if STOPS_FILE.exists():
    stops_raw = pd.read_csv(STOPS_FILE, sep=";", encoding="utf-8-sig", low_memory=False)
else:
    # Einmaliger Download, danach wird die lokale Kopie verwendet
    stops_raw = pd.read_csv(STOPS_URL, sep=";", encoding="utf-8-sig", low_memory=False)
    stops_raw.to_csv(STOPS_FILE, sep=";", index=False, encoding="utf-8-sig")

print(stops_raw.shape)
stops_raw[["Haltestelle SLOID", "offizielle Haltestellen Bezeichnung", "geopos_haltestelle"]].head()

In [ ]:
# Die Koordinaten stehen als Text "46.55, 6.61" in einer Spalte. Für die Karte brauchen wir zwei Zahlen.
coords = stops_raw["geopos_haltestelle"].str.split(",", expand=True).astype(float)

# Nur die Spalten behalten, die wir später brauchen
stops = pd.DataFrame({
    "sloid": stops_raw["Haltestelle SLOID"],
    "name": stops_raw["offizielle Haltestellen Bezeichnung"].str.strip(),
    "lat": coords[0],
    "lon": coords[1],
})

# Ohne Koordinaten ist eine Haltestelle für die Karte nutzlos.
# Die Datei hat eine Zeile pro Haltekante (Gleis, Perron), wir wollen eine Zeile pro Haltestelle.
stops = stops.dropna(subset=["lat", "lon"])
stops = stops.drop_duplicates(subset="sloid")
print("Haltestellen:", len(stops))
print("Namen eindeutig:", stops["name"].is_unique)

## 1.2 Kontrolldaten einlesen

Jede Zeile ist eine Kontrolle, bei der jemand ohne gültigen Fahrausweis angetroffen wurde. Die Datei enthält Geburtsdaten und ist deshalb nicht im Repository. Sie muss lokal unter `data/raw/ASP_202509.XLSX` liegen.

`pd.read_excel` liest die Excel-Datei ebenfalls in ein **pandas DataFrame** (`controls_raw`).

In [ ]:
ASP_FILE = RAW / "ASP_202509.XLSX"
if not ASP_FILE.exists():
    raise FileNotFoundError("ASP_202509.XLSX fehlt. Bitte lokal in data/raw ablegen (siehe README).")

controls_raw = pd.read_excel(ASP_FILE)
print(controls_raw.shape)
controls_raw.head()

## 1.3 Daten prüfen

Bevor wir bereinigen, schauen wir, wo die Probleme liegen.

In [ ]:
print("Fehlende Werte pro Spalte:")
print(controls_raw.isna().sum())
print()
print("Doppelte Zeilen:", controls_raw.duplicated().sum())
print("Start gleich Ziel:", (controls_raw["STATION_FROM"] == controls_raw["STATION_TO"]).sum())

In [ ]:
# Der Export ist vom September 2025, enthält aber auch ältere Kontrollen
controls_raw["CONTROL_DATE"].str[:7].value_counts().sort_index().tail(8)

In [ ]:
# Die Verkehrsunternehmen sind nur als Nummer erfasst
print("Anzahl Unternehmen:", controls_raw["DRIVING_COMPANY"].nunique())
controls_raw["DRIVING_COMPANY"].value_counts().head()

In [ ]:
# Der Grund ist Freitext in Deutsch, Französisch und Italienisch
print("Verschiedene Gründe:", controls_raw["REASON"].nunique())
controls_raw["REASON"].value_counts().head(10)

## 1.4 Bereinigen

Alle Schritte arbeiten mit Methoden des **pandas DataFrame** (zum Beispiel `drop_duplicates`, `to_datetime`, `cut`, `apply`). Das Ergebnis ist das bereinigte DataFrame `controls`.

Was wir tun und warum:

- **Duplikate entfernen**, sonst zählen wir Fälle doppelt.
- **Datum und Zeit zusammenführen**, damit wir Wochentag und Stunde auswerten können.
- **Geburtsdatum durch Jahrgang und Altersgruppe ersetzen.** Das volle Geburtsdatum könnte zusammen mit Ort und Zeit eine Person erkennbar machen.
- **Gründe in wenige Kategorien zusammenfassen**, statt über 500 verschiedene Texte.
- **Unternehmensnummern durch Namen ersetzen**, damit die Diagramme lesbar sind.
- **Geschlecht als Text** statt Zahlencode.

In [ ]:
# Doppelte Zeilen entfernen
controls = controls_raw.drop_duplicates().copy()
print("Zeilen vorher:", len(controls_raw), "nachher:", len(controls))

# Datum und Zeit zu einem Zeitpunkt zusammenführen, dann Wochentag (0 = Montag) und Stunde ableiten
controls["datetime"] = pd.to_datetime(controls["CONTROL_DATE"] + " " + controls["CONTROL_TIME"])
controls["weekday"] = controls["datetime"].dt.day_of_week
controls["hour"] = controls["datetime"].dt.hour

In [ ]:
# Jahrgang statt Geburtsdatum. "Int64" erlaubt ganze Zahlen auch bei fehlenden Werten.
birthdate = pd.to_datetime(controls["BIRTHDATE"])
controls["birth_year"] = birthdate.dt.year.astype("Int64")

# Alter zum Zeitpunkt der Kontrolle, danach in Gruppen einteilen
age = (controls["datetime"] - birthdate).dt.days // 365
controls["age_group"] = pd.cut(age, bins=[0, 17, 25, 35, 50, 65, 120],
                               labels=["unter 18", "18 bis 25", "26 bis 35", "36 bis 50", "51 bis 65", "über 65"])
controls["age_group"].value_counts().sort_index()

In [ ]:
# Gründe nach Stichworten einteilen. Die erste passende Kategorie gewinnt.
categories = {
    "E-Ticket / Check-In": ["e-ticket", "check-in", "checkin", "e-billet"],
    "Abo abgelaufen": ["abo", "abonnement", "abgelaufen", "expir"],
    "Zone / Strecke fehlt": ["zone", "haltestelle", "arrêt", "arret", "strecke", "trajet", "fermata"],
    "Falsche Klasse / Kundengruppe": ["klasse", "kundengruppe", "classe", "ermäss", "réduit", "reduit"],
    "Kein gültiges Billett": ["ohne g", "kein", "sans titre", "senza titolo", "billett", "fahrausweis",
                              "titre de transport", "biglietto"],
}


def categorize(reason):
    text = str(reason).lower()
    for category, keywords in categories.items():
        for keyword in keywords:
            if keyword in text:
                return category
    return "Andere"


controls["reason_cat"] = controls["REASON"].apply(categorize)
controls["reason_cat"].value_counts()

In [ ]:
# Kontrolle: Was bleibt in "Andere"? Meist Abkürzungen, die wir nicht sicher deuten können.
controls[controls["reason_cat"] == "Andere"]["REASON"].value_counts().head(10)

In [ ]:
# Die Haltestellendaten enthalten zu jeder Unternehmensnummer ein Kürzel
names = stops_raw[["businessOrganisationNumber", "businessOrganisationAbbreviationDe"]].dropna()
names = names.drop_duplicates(subset="businessOrganisationNumber")
company_names = dict(zip(names["businessOrganisationNumber"], names["businessOrganisationAbbreviationDe"]))

# Von Hand ergänzt: 801 ist PostAuto (das Kürzel PAG kennt kaum jemand).
# 490 vermuten wir als ZVV, weil fast alle Fälle im Raum Zürich liegen. Das ist nicht bestätigt.
company_names[801] = "PostAuto"
company_names[490] = "ZVV (?)"


def company_name(number):
    name = company_names.get(number, "Unbekannt")
    # Doppelte Kürzel wie "BLS-bls" kürzen wir auf "BLS"
    parts = name.split("-")
    if len(parts) == 2 and parts[0].lower() == parts[1].lower():
        return parts[0]
    return name


controls["company"] = controls["DRIVING_COMPANY"].apply(company_name)
print("Anteil Unbekannt:", round((controls["company"] == "Unbekannt").mean() * 100, 1), "%")
controls["company"].value_counts().head(10)

In [ ]:
# Der Code für das Geschlecht ist nicht dokumentiert. Wir nehmen die Norm ISO 5218 an
# (1 = männlich, 2 = weiblich). Das Fragezeichen zeigt, dass es nicht bestätigt ist.
gender_labels = {1: "männlich (?)", 2: "weiblich (?)", 3: "andere (?)"}
controls["gender"] = controls["GENDER"].map(gender_labels).fillna("unbekannt")
controls["gender"].value_counts()

## 1.5 Haltestellen zuordnen

Wir verbinden Start- und Zielhaltestelle über den Namen mit den Haltestellendaten. So erhält jede Kontrolle Koordinaten für die Karte. Dazu nutzen wir `map` auf den Spalten des DataFrames.

In [ ]:
# Leerzeichen am Rand entfernen, sonst passen die Namen nicht zusammen
controls["STATION_FROM"] = controls["STATION_FROM"].str.strip()
controls["STATION_TO"] = controls["STATION_TO"].str.strip()

lat_by_name = stops.set_index("name")["lat"]
lon_by_name = stops.set_index("name")["lon"]

controls["lat_from"] = controls["STATION_FROM"].map(lat_by_name)
controls["lon_from"] = controls["STATION_FROM"].map(lon_by_name)
controls["lat_to"] = controls["STATION_TO"].map(lat_by_name)
controls["lon_to"] = controls["STATION_TO"].map(lon_by_name)

print("Start gefunden:", round(controls["lat_from"].notna().mean() * 100, 1), "%")
print("Ziel gefunden:", round(controls["lat_to"].notna().mean() * 100, 1), "%")

In [ ]:
# Nicht gefunden werden vor allem Bahnhöfe im Ausland und abweichende Schreibweisen
controls[controls["lat_from"].isna()]["STATION_FROM"].value_counts().head(10)

## 1.6 Bereinigte Daten speichern

Wir speichern nur die Spalten, die wir für die Analyse brauchen. Das Geburtsdatum ist nicht dabei. `to_csv` schreibt das DataFrame als CSV-Datei.

In [ ]:
columns = ["datetime", "weekday", "hour", "company", "LINE", "STATION_FROM", "STATION_TO",
           "gender", "birth_year", "age_group", "reason_cat", "lat_from", "lon_from", "lat_to", "lon_to"]
controls_clean = controls[columns].rename(columns={"LINE": "line", "STATION_FROM": "station_from",
                                                  "STATION_TO": "station_to"})

controls_clean.to_csv(PROCESSED / "controls_clean.csv", index=False)
print("Gespeichert:", len(controls_clean), "Zeilen")
controls_clean.head()